In [5]:

import os
import argparse
from datetime import datetime
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.linear_model import LogisticRegression, Lasso, LinearRegression, Ridge
from sklearn.dummy import DummyClassifier, DummyRegressor
from collections import defaultdict
from sklearn.metrics import (
    mean_squared_error,
    accuracy_score,
    roc_auc_score,
    balanced_accuracy_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pickle

from cache.cache_utils import load_saved_data
from src.utils import *

In [2]:
DATASET_NAMES = ["mmlu_high_school", "mmlu_professional", "ARC-Challenge", "ARC-Easy", "sms_spam"]
DATASET_TITLES = ["MMLU High School", "MMLU Professional", "ARC Challenge", "ARC Easy", "SMS Spam"]
MODEL_NAMES = ["Apertus-8B-Instruct-2509", "Llama-3.1-8B-Instruct"]
INDEX = 0
SELECTED_DATA_INDEX = [1, 2]

In [3]:
TRANSFORM_TARGETS = True
NORMALIZE_FEATURES = True
NR_LAYERS = 32
SELECTED_DATASETS = [DATASET_NAMES[i] for i in SELECTED_DATA_INDEX]
MODEL_NAME = MODEL_NAMES[INDEX]
SAVE_NAME = ""
SEED = 52
SAVE_CACHE_KEY = ""
SAVE_DIR = "/users/astepancic/projects/apertus-probes/scratch/mera-runs"
NR_ATTEMPTS = 5 # for CV
MAX_TRIALS = 5 # for non-convergence retries
ERROR_TYPE = "SM" # SM or CE (squared error or cross-entropy)
EPSILON = 1e-10 # for counting non-zero coefficients



In [3]:
METRICS ={"classification": {"AUCROC": roc_auc_score, "ACC": accuracy_score, "BACC": balanced_accuracy_score},\
              "regression": {"MSE": mean_squared_error} }

probe_results = []

# FIXME: there is no SEED in cuml models
def initialise_regression_models(seed: int, alphas) -> dict:
    """Initialise regression models with various hyperparameters."""
    models = {
        f"L-{alpha}": Lasso(
                alpha=alpha, fit_intercept=False, max_iter=2000, random_state=seed
        ) for alpha in alphas
    }  # positive=True,
    models["L-0"] = LinearRegression(fit_intercept=False, n_jobs=5)
    return models

def initalise_classification_models(seed: int) -> dict:
        models = {
            "LogReg-l1": LogisticRegression(
                penalty="l1",
                solver="liblinear",
                max_iter=2000,
                fit_intercept=False,
                random_state=seed,
            )
        }
        return models

alphas = [0.5, 0.25, 0.1, 0.05]
MODELS = {
    "classification": initalise_classification_models(SEED),
    "regression": initialise_regression_models(SEED, alphas),
}

In [ ]:
def merge_activations(activations_list):
    merged = {}
    all_layers = list(activations_list[0].keys())
    for layer in all_layers:
        arrays = [activations[layer] for activations in activations_list]
        merged[layer] = np.concatenate(arrays, axis=0)
    return merged


def load_datasets(dataset_names, model_name, save_dir):
    y_error_last_list = []
    y_correct_last = []
    y_error_exact_list = []
    y_correct_exact = []
    activations_exact_list = []
    activations_last_list = []
    for dataset_name in dataset_names:
        results_path = f'{save_dir}/{dataset_name}/{model_name}/results.pkl'
        os.makedirs(os.path.dirname(results_path), exist_ok=True)

        activations_path = f'{save_dir}/{dataset_name}/{model_name}/acts.pkl'
        print(activations_path)
        with open(activations_path, 'rb') as f:
            activations = pickle.load(f)

        activations_last_list.append(activations["activations_cache"])
        y_error_last_list.append(activations["y_error_sm"] if ERROR_TYPE == "SM" else activations["y_error_ce"])
        y_correct_last.extend(activations["y_correct"])

        activations_exact_list.append(activations["activations_cache_exact"])
        y_error_exact_list.append(activations["y_error_sm_exact"] if ERROR_TYPE == "SM" else activations["y_error_ce_exact"])
        y_correct_exact.extend(activations["y_correct_exact"])

    y_error_exact = np.concatenate(y_error_exact_list, axis=0)
    y_error_last = np.concatenate(y_error_last_list, axis=0)
    activations_exact = merge_activations(activations_exact_list)
    activations_last = merge_activations(activations_last_list)

    return y_error_exact, y_correct_exact, activations_exact, y_error_last, y_correct_last, activations_last

SAVE_RESULTS_PATH = f'{SAVE_DIR}/mix/{MODEL_NAME}/df_probes_{SAVE_NAME}.pkl'
print(f"Saving results to {SAVE_RESULTS_PATH}")

y_error_exact, y_correct_exact, activations_exact, y_error_last, y_correct_last, activations_last = load_datasets(SELECTED_DATASETS, MODEL_NAME, SAVE_DIR)

# do some tests of what is in y_error_exact, y_correct_exact
print(f"y_error_exact shape: {y_error_exact.shape}")
print(f"y_correct_exact length: {len(y_correct_exact)}")
print(f"activations_exact keys: {list(activations_exact.keys())}")


Saving results to /users/astepancic/projects/apertus-probes/scratch/mera-runs/mix/Apertus-8B-Instruct-2509/df_probes_.pkl
/users/astepancic/projects/apertus-probes/scratch/mera-runs/mmlu_professional/Apertus-8B-Instruct-2509/acts.pkl
/users/astepancic/projects/apertus-probes/scratch/mera-runs/ARC-Challenge/Apertus-8B-Instruct-2509/acts.pkl
y_error_exact shape: (5550,)
y_correct_exact length: 5550
activations_exact keys: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31]
(5550, 4096)


The `activations` variable is a dictionary that contains the following keys:
- `activations_cache`: A dictionary where each key is an integer (layer index) and the value is a NumPy array representing the activations for that layer.
- `y_correct`: A list of boolean values indicating whether the predictions were correct for each sample.
- `y_error_sm`: A NumPy array representing the softmax error for each sample.
- `y_error_ce`: A list of cross-entropy errors for each sample.
- `activations_cache_exact`: Similar to `activations_cache`, but for exact matches.
- `y_correct_exact`: A list of boolean values indicating whether the predictions were correct for exact matches.
- `y_error_sm_exact`: A NumPy array representing the softmax error for exact matches.
- `y_error_ce_exact`: A list of cross-entropy errors for exact matches.

### Example: Accessing the activations for layer 0


In [ ]:
def train_model(task_type, model_dict, model_name, model, layer_idx, layer_data, probe_results):
    print("Layer", layer_idx)
    print(layer_data.shape)
    assert isinstance(layer_data, np.ndarray)
    X = layer_data
    if NORMALIZE_FEATURES:
        print("normalizeing features")
        X = StandardScaler().fit_transform(X)
        print("Feature means (first 5):", X.mean(axis=0)[:5])
        print("Feature stds (first 5):", X.std(axis=0)[:5])
    if task_type == "classification":
        y_true = y_correct_exact
    elif task_type == "regression":
        y_true = y_error_exact
        if TRANSFORM_TARGETS and task_type == "regression":
            y_true = np.clip(y_true, 1e-8, 1 - 1e-8)
            y_true = np.log(y_true / (1 - y_true))
    else:
        raise ValueError(f"Unknown task type: {task_type}")
    # Transform to log-odd-ratio space
    

    # FIXME Why use normalise error?
    #   elif normalise_error and model_task == "regression":
    #         y_true /= y_true.max()

    
    for m in range(NR_ATTEMPTS):
        print("Layer:", layer_idx, "Attempt:", m)
        
        X_train, X_test, y_train, y_test = train_test_split(
            X, y_true, test_size=0.3, random_state=SEED + m
        )
        # print number of parameters in X_train and number of samples

        dummy_model = (
            DummyClassifier(strategy="most_frequent")
            if task_type == "classification"
            else DummyRegressor(strategy="mean")
        ) 
        dummy_model.fit(X_train, y_train)
        dummy_y_pred = dummy_model.predict(X_test)
        dummy_metrics = {
            f"Dummy-{metric_name}": metric(
                y_test, dummy_y_pred
            )
            for metric_name, metric in METRICS[task_type].items()
        }
        no_coeffs = True
        trials = 0
        while no_coeffs and trials < MAX_TRIALS:
            trials += 1
            model.fit(X_train, y_train)
            coef_norm = np.linalg.norm(model.coef_)
            s = np.linalg.svd(X_train, compute_uv=False)
            quantile = np.quantile(s, [0, 0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99, 1.0])
            cond_est = s[0] / s[-1] if s[-1] > 0 else np.inf
            # print("coef_norm:", coef_norm, "cond_est:", cond_est)
            # print("||w||:", coef_norm)  # track vs. layer; exploding norms → near-separable/hard
            # print("n_iter_:", getattr(model, "n_iter_", None))  # per class for OvR
            # print(f"cond_est≈{cond_est:.1e}, y_balance={np.mean(y_train):.3f}") # track vs. layer; high cond_est → near-separable/hard

            y_pred = model.predict(X_test)

            # Collect coefficients and count non-zero ones.
            assert hasattr(model, "coef_") # using sklearn's LogisticRegression
            non_zero_coeffs = np.where(np.abs(model.coef_) > EPSILON)[0]
            no_coeffs = len(non_zero_coeffs) == 0
        if no_coeffs:
            print(f"Warning: No non-zero coefficients found after {MAX_TRIALS} trials at layer {layer_idx}, attempt {m}.")
        
        results = {
            **{
                metric_name: metric(y_test, y_pred)
                for metric_name, metric in METRICS[task_type].items()
            },
            **dummy_metrics,
        }
        print(model_name, layer_idx, "Results:", results)

        if not no_coeffs:
            result_line = {
                "Dataset": DATASET_NAME,
                "LLM_Model": MODEL_NAME,
                "Task": task_type,
                "Probe Model": model_name,
                "Layer": layer_idx,
                "Coefficients": model.coef_.flatten(),
                "Coef-norm": coef_norm,
                "Cond-Est": cond_est,
                "# of non-zero coefficients": len(non_zero_coeffs),
                "Attempt": m,
                "Token-Pos": "exact",
                "y_pred": y_pred,
                "y_test": y_test,
                **results,
            }
            probe_results.append(result_line)

In [ ]:
import multiprocessing as mp
from concurrent.futures import ThreadPoolExecutor, as_completed
with ThreadPoolExecutor(max_workers=25) as executor:
    futures = []
    for task_type, model_dict in MODELS.items():
        print("Task type:", task_type)
        for model_name, model in model_dict.items():
            print("Model:", model_name)
            for layer_idx, layer_data in tqdm(activations.items(), desc="Processing Features"):
                futures.append(
                    executor.submit(
                        train_model,
                        task_type,
                        model_dict,
                        model_name,
                        model,
                        layer_idx,
                        layer_data,
                        probe_results
                    )
                )    

for fut in tqdm(as_completed(futures), total=len(futures), desc="Training probes"):
    ds = fut.result()
    print(f"[INFO] Finished {ds}")
df_results = pd.DataFrame(probe_results)
            

Task type: classification
Model: LogReg-l1


Processing Features: 100%|██████████| 32/32 [00:00<00:00, 3151.24it/s]


Layer 0
(3860, 4096)
normalizeing features
Layer 1
(3860, 4096)
normalizeing features
Layer 2
(3860, 4096)
normalizeing features
Layer 3
(3860, 4096)
normalizeing features
Layer 4
(3860, 4096)
normalizeing features
Layer 5
(3860, 4096)
normalizeing features
Layer 6
(3860, 4096)
normalizeing features
Layer 7
(3860, 4096)
normalizeing features
Layer 8
(3860, 4096)
normalizeing features
Layer 9
(3860, 4096)
normalizeing features
Layer 10
(3860, 4096)
normalizeing features
Layer 11
(3860, 4096)
normalizeing features
Layer 12
(3860, 4096)
normalizeing features
Layer 13
(3860, 4096)
normalizeing features
Layer 14
(3860, 4096)
normalizeing features
Layer 15
(3860, 4096)
normalizeing features
Layer 16
(3860, 4096)
normalizeing features
Layer 17
(3860, 4096)
normalizeing features
Layer 18
(3860, 4096)
normalizeing features
Layer 19
(3860, 4096)
normalizeing features
Layer 20
(3860, 4096)
normalizeing features
Layer 21
(3860, 4096)
normalizeing features
Layer 22
(3860, 4096)
normalizeing feature

Processing Features: 100%|██████████| 32/32 [00:00<00:00, 91616.20it/s]


Model: L-0.25


...

Feature means (first 5): [-6.04076092e-07  9.98455008e-08 -1.08739876e-07  1.08786196e-07
  3.08832355e-08]
Feature means (first 5): [ 1.2387265e-07 -2.1649148e-08  7.6640610e-08 -1.9456439e-09
 -2.4938213e-08]
Feature means (first 5): [ 3.1809733e-09  1.4453354e-08  1.4499679e-08 -3.5951945e-08
  4.3591687e-08]
Feature means (first 5): [-5.7471770e-08 -1.6769597e-08 -1.7286891e-07 -3.1686199e-08
 -1.0098818e-08]
Feature means (first 5): [-1.8541522e-08 -1.0037051e-08 -1.9196825e-08  5.3041957e-09
  9.0101837e-09]
Feature means (first 5): [ 6.0897882e-09  5.0648508e-09  6.9262413e-08  7.5239285e-08
 -9.7098820e-08]
Feature means (first 5): [ 1.9193932e-08  1.7572560e-08  4.4867550e-09 -1.2159309e-07
 -1.1117964e-09]
Feature means (first 5): [-4.7374883e-08  1.8931424e-08  1.7186134e-07 -2.8505227e-08
 -3.9020968e-08]
Feature means (first 5): [-2.5710294e-08  2.1587383e-08 -1.4329822e-08  2.2992570e-08
 -6.3436097e-08]
Feature means (first 5): [ 1.40518726e-08  1.03443398e-07  1.2021299

Feature stds (first 5): [1.0000001  1.0000004  0.9999993  0.99999946 0.9999993 ]
Layer: 20 Attempt: 0
Feature stds (first 5): [1.0000002  0.99999976 0.99999946 0.99999994 1.        ]
Layer: 22 Attempt: 0
Feature stds (first 5): [0.9999999 0.9999999 1.0000004 1.0000001 1.0000001]
Layer: 19 Attempt: 0
Feature stds (first 5): [0.99999875 0.9999996  0.9999995  1.0000001  1.0000007 ]
Layer: 15 Attempt: 0
Feature stds (first 5): [0.99999976 1.0000006  0.99999976 1.0000004  0.99999994]
Layer: 24 Attempt: 0
Feature stds (first 5): [0.9999997  1.0000005  0.99999964 0.9999996  1.        ]
Layer: 18 Attempt: 0
Feature stds (first 5): [1.0000004 0.9999998 0.9999995 1.0000004 1.0000001]
Layer: 12 Attempt: 0
Feature stds (first 5): [0.99999934 1.0000001  1.0000004  0.9999999  1.0000002 ]
Layer: 6 Attempt: 0
Feature stds (first 5): [1.0000001  0.99999934 0.99999905 0.99999934 0.9999998 ]
Layer: 23 Attempt: 0
Feature stds (first 5): [0.99999964 0.99999994 0.9999997  1.0000002  0.9999996 ]
Layer: 21 At

Training probes:   1%|          | 1/192 [00:00<00:00, 2723.57it/s]

[INFO] Finished None


AttributeError: 'int' object has no attribute 'append'

In [ ]:
df_results.to_pickle(SAVE_RESULTS_PATH + '.pkl')

NameError: name 'df_results' is not defined

In [ ]:
# I want to create essential results and save it to csv.
essential_results = df_results[[
    "Dataset", "LLM_Model", "Task", "Probe Model", "Error-Type", "Layer",
    "# of non-zero coefficients", "Attempt", "Token-Pos"
] + list(METRICS.keys())]
essential_results.to_csv(SAVE_RESULTS_PATH + '.csv', index=False)
essential_results.head()

,Dataset,LLM_Model,Task,Probe Model,Error-Type,Layer,# of non-zero coefficients,Attempt,Token-Pos,AUCROC,ACC,BACC
0,mmlu_high_school,swiss-ai/Apertus-8B-Instruct-2509,Classification,LogisticRegression-L2,Exact,0,4096,0,exact,0.570515,0.605556,0.570515
1,mmlu_high_school,swiss-ai/Apertus-8B-Instruct-2509,Classification,LogisticRegression-L2,Exact,0,4096,1,exact,0.555762,0.608889,0.555762
2,mmlu_high_school,swiss-ai/Apertus-8B-Instruct-2509,Classification,LogisticRegression-L2,Exact,0,4096,2,exact,0.561055,0.596667,0.561055
3,mmlu_high_school,swiss-ai/Apertus-8B-Instruct-2509,Classification,LogisticRegression-L2,Exact,0,4096,3,exact,0.572098,0.615556,0.572098
4,mmlu_high_school,swiss-ai/Apertus-8B-Instruct-2509,Classification,LogisticRegression-L2,Exact,0,4096,4,exact,0.561118,0.610000,0.561118


In [ ]:
completions_path = f'{SAVE_DIR}/{DATASET_NAME}/{MODEL_NAME.split("/")[1]}/{SAVE_CACHE_KEY}_completions.pkl'

with open(completions_path, "rb") as f:
    completions = pickle.load(f)
# print(completions.keys())
# print(completions["completions"][:2])
# print(completions["answers"][:2])
# Assuming `completions` contains a list of token sequences and `match_indices` contains indices
match_indices = completions["match_indices"]
generations = completions["completions"]  # Assuming `tokens` contains the token sequences
# Inspect the tokens at `match_indices` and surrounding tokens
for i, (generation, match_index) in enumerate(zip(generations, match_indices)):
    generation = generation.squeeze(0)
    if match_index < len(generation):  # Ensure the index is within bounds
        print(f"Sample {i}:")
        print("Generation:", len(generation))
        print("Match index: ", match_index)
        print(f"Matched Token: {generation[match_index]}")
        # Print tokens around the matched token (e.g., 5 tokens before and after)
        start = max(0, match_index - 5)
        end = min(len(tokens[i]), match_index + 6)
        print(f"Context: {' '.join(tokens[i][start:end])}")
        print("-" * 50)
    else:
        print(f"Sample {i}: Match index {match_index} is out of bounds for tokens.")


Sample 0:
Generation: 203
Match index:  105
Matched Token: 1349
Context: 
--------------------------------------------------
Sample 1:
Generation: 206
Match index:  106
Matched Token: 1407
Context: 
--------------------------------------------------
Sample 2:
Generation: 175
Match index:  75
Matched Token: 1359
Context: 
--------------------------------------------------
Sample 3:
Generation: 179
Match index:  79
Matched Token: 1407
Context: 
--------------------------------------------------
Sample 4:
Generation: 210
Match index:  110
Matched Token: 1349
Context: 
--------------------------------------------------
Sample 5:
Generation: 192
Match index:  92
Matched Token: 1349
Context: 
--------------------------------------------------
Sample 6:
Generation: 145
Match index:  73
Matched Token: 1398
Context: 
--------------------------------------------------
Sample 7:
Generation: 174
Match index:  74
Matched Token: 1398
Context: 
--------------------------------------------------
Sampl

TypeError: sequence item 0: expected str instance, numpy.ndarray found